# P12026 Part 2: Pandas for NLP Preprocessing

This notebook teaches how to use Pandas to prepare text data for NLP tasks.

By the end of this section, you will be able to:
- create and inspect a DataFrame
- remove duplicate rows
- clean and transform text columns
- apply useful Pandas functions before tokenization and modeling

## Section 1: Pandas Basics for Text Data

Pandas is a Python library for tabular data analysis. In NLP workflows, it is commonly used to manage datasets of documents, tweets, reviews, and labels.

### What is a DataFrame?

A DataFrame is a two-dimensional table with rows and columns.

In this example:
- each row is one text record
- columns store attributes such as the text, source, and label

In [1]:
import pandas as pd

data = {
    "tweet": [
        "Hello WORLD!",
        "Pandas is GREAT",
        "Python is FUN",
        "Hello WORLD!",
        "",
        "NLP with pandas is useful"
    ],
    "source": ["x", "x", "y", "x", "y","y"],
    "label": ["greeting", "statement", "statement", "greeting", "statement","statement"]
}

df = pd.DataFrame(data)
df

,tweet,source,label
0,Hello WORLD!,x,greeting
1,Pandas is GREAT,x,statement
2,Python is FUN,y,statement
3,Hello WORLD!,x,greeting
4,,y,statement
5,NLP with pandas is useful,y,statement


### Quick DataFrame Inspection

Before cleaning text, it is a good habit to inspect the table shape, columns, and first rows.

In [8]:
print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head()

Shape: (6, 3)
Columns: ['tweet', 'source', 'label']


,tweet,source,label
0,Hello WORLD!,x,greeting
1,Pandas is GREAT,x,statement
2,Python is FUN,y,statement
3,Hello WORLD!,x,greeting
4,,y,statement


### Dropping Duplicates

Duplicate text rows can bias analysis and model training.

Use `drop_duplicates()` to keep one copy of repeated rows.

In [9]:
print("Rows before deduplication:", len(df))
df_no_duplicates = df.drop_duplicates(subset=["tweet"]).copy()
print("Rows after deduplication:", len(df_no_duplicates))
df_no_duplicates

Rows before deduplication: 6
Rows after deduplication: 5


,tweet,source,label
0,Hello WORLD!,x,greeting
1,Pandas is GREAT,x,statement
2,Python is FUN,y,statement
4,,y,statement
5,NLP with pandas is useful,y,statement


### Other Useful Pandas Functions for NLP task

These functions are commonly used before tokenization:
- `str.lower()` to normalize letter case
- `str.replace(..., regex=True)` to remove punctuation
- `str.strip()` to remove extra spaces
- `replace(r"^\\s*$", pd.NA, regex=True)` to convert empty/whitespace-only text into missing values
- `dropna()` to remove rows with missing text values
- `value_counts()` to inspect class balance

In [10]:
clean_df = df_no_duplicates.copy()

clean_df["tweet"] = (
    clean_df["tweet"]
    .str.lower()
    .str.replace(r"[^\w\s]", "", regex=True)
    .str.strip()
)

# Convert empty strings (or whitespace-only strings) to missing values.
clean_df["tweet"] = clean_df["tweet"].replace(r"^\s*$", pd.NA, regex=True)

clean_df = clean_df.dropna(subset=["tweet"])
clean_df

,tweet,source,label
0,hello world,x,greeting
1,pandas is great,x,statement
2,python is fun,y,statement
5,nlp with pandas is useful,y,statement


### Label Distribution Check

Checking label counts helps you understand if one class dominates the dataset.

In [11]:
clean_df["label"].value_counts()

label
statement    3
greeting     1
Name: count, dtype: int64

## Section 2: Text Preprocessing Pipeline

This section groups multiple preprocessing steps that are commonly applied before NLP modeling.

Current steps in this section:
- convert text to lowercase
- remove punctuation
- remove stop words
- standardize text
- correct spelling with TextBlob and pyspellchecker
- tokenize text with TextBlob
- apply stemming with NLTK
- apply lemmatization with TextBlob (and NLTK)


In [12]:
lambda_df = df_no_duplicates.copy()

# Convert each tweet to lowercase using a lambda expression
# split() -> lowercase each word -> join back into one string
lambda_df["tweet"] = lambda_df["tweet"].apply(
    lambda tweet: " ".join(word.lower() for word in tweet.split())
)

lambda_df[["tweet", "label"]]

,tweet,label
0,hello world!,greeting
1,pandas is great,statement
2,python is fun,statement
4,,statement
5,nlp with pandas is useful,statement


### Step 1: Lowercase Conversion (Lambda)

Explanation of the lambda expression:

- `df['tweet'].apply(...)`: applies a function to each row in the `tweet` column.
- `lambda tweet: ...`: defines a short function for each tweet.
- `tweet.split()`: splits the tweet string into words.
- `word.lower() for word in tweet.split()`: lowercases each word.
- `' '.join(...)`: joins the lowercase words back into a single sentence.

### Step 2: Remove Punctuation

Punctuation can add noise in NLP tasks (especially for bag-of-words or frequency-based models).

In this step, we remove punctuation while keeping letters, numbers, and spaces.

In [13]:
punctuation_df = lambda_df.copy()

# Remove punctuation using regex:
# [^\w\s] matches any character that is NOT a word character and NOT whitespace.
punctuation_df["tweet"] = punctuation_df["tweet"].str.replace(r"[^\w\s]", "", regex=True)

punctuation_df[["tweet", "label"]]

,tweet,label
0,hello world,greeting
1,pandas is great,statement
2,python is fun,statement
4,,statement
5,nlp with pandas is useful,statement


Why this regex works:

- `\w` = letters, digits, and underscore
- `\s` = whitespace (spaces, tabs, new lines)
- `[^\w\s]` = anything that is not a word character and not whitespace

So punctuation marks like `. , ! ? : ;` are removed.

In [14]:
# Alternative method using Python's built-in punctuation list
import string

alt_text = "I. like, this book!"
for c in string.punctuation:
    alt_text = alt_text.replace(c, "")

alt_text

'I like this book'

### Step 3: Stop Word Removal

Stop words are very common words in a language that usually carry less meaning in many NLP tasks.

Examples include words such as:
- `the`
- `is`
- `and`
- `to`

Why remove stop words:
- It reduces noise in the text.
- It helps models focus on more informative words.
- It can improve efficiency by reducing the number of tokens.

Important note:
- In some tasks, you may want to keep stop words. For example, machine translation, question answering, or sentiment analysis may benefit from preserving them.

### Using NLTK for Stop Words

`NLTK` stands for `Natural Language Toolkit`.

It is a popular Python library for working with human language data. It provides tools for:
- tokenization
- stop words
- stemming
- lemmatization
- corpora and linguistic resources

Why do we sometimes need `nltk.download(...)`?
- Some NLTK features depend on external language resources.
- These resources are not always installed automatically with the package.
- You usually download them once per environment, and then reuse them later.

For stop-word removal, the resource we need is usually `stopwords`.

In [15]:
import nltk
from nltk.corpus import stopwords

# Download the stopwords resource only if it is missing.
try:
    nltk.data.find("corpora/stopwords")
except LookupError:
    nltk.download("stopwords")

stopword_df = punctuation_df.copy()
stop_words = stopwords.words("english")

stopword_df["tweet"] = stopword_df["tweet"].apply(
    lambda tweet: " ".join(word for word in tweet.split() if word.lower() not in stop_words)
)

stopword_df[["tweet", "label"]]

,tweet,label
0,hello world,greeting
1,pandas great,statement
2,python fun,statement
4,,statement
5,nlp pandas useful,statement


### Step 4: Text Standardization

Text standardization means converting text into a more consistent and normalized form.

Why this is useful in NLP:
- It reduces variation caused by abbreviations, slang, or informal writing.
- It helps similar expressions map to the same meaning.
- It improves downstream tasks such as search, classification, and analysis.

Examples:
- `nlp` -> `natural language processing`
- `ur` -> `your`
- `wbu` -> `what about you`

In the next section we will:
- split text by whitespace
- remove punctuation from each word
- check the lowercase version against the lookup dictionary
- if found, replace it with the mapped full form
- append the result
- join words back into a sentence

In [16]:
lookup_dict = {
    "nlp": "natural language processing",
    "ur": "your",
    "wbu": "what about you"
}

import re

def text_standardize(input_text):
    words = input_text.split()
    standardized_words = []

    for word in words:
        cleaned_word = re.sub(r"[^\w\s]", "", word)
        if cleaned_word.lower() in lookup_dict:
            cleaned_word = lookup_dict[cleaned_word.lower()]
        standardized_words.append(cleaned_word)

    return " ".join(standardized_words)

standardized_df = stopword_df.copy()
standardized_df["tweet"] = standardized_df["tweet"].apply(text_standardize)
standardized_df[["tweet", "label"]]

,tweet,label
0,hello world,greeting
1,pandas great,statement
2,python fun,statement
4,,statement
5,natural language processing pandas useful,statement


### Step 5: Spelling Correction with TextBlob

`TextBlob` is a Python library for working with textual data.

It offers a simple interface for common NLP tasks such as:
- part-of-speech tagging
- noun phrase extraction
- sentiment analysis
- spelling correction

In this step, we use TextBlob to automatically correct misspelled words. This can help make the text cleaner before later NLP tasks, although it should be used carefully because automatic correction is not always perfect.

If `TextBlob` is not installed in your environment, install it first with `pip install textblob`.

In [5]:
from textblob import TextBlob

spelling_examples = pd.DataFrame(
    {
        "original_text": [
            "Introdaction to NLP",
            "Machin learning is the new electrcity",
            "R is good langauage",
            "I want more books like ths"
        ]
    }
)

spelling_examples["corrected_text"] = spelling_examples["original_text"].apply(
    lambda text: str(TextBlob(text).correct())
)

spelling_examples

,original_text,corrected_text
0,Introdaction to NLP,Introduction to NLP
1,Machin learning is the new electrcity,Machine learning is the new electricity
2,R is good langauage,R is good language
3,I want more books like ths,I want more books like the


### Spell Correction with pyspellchecker

`pyspellchecker` is a lightweight Python library for detecting and correcting misspelled words.

It works by comparing words against a built-in dictionary and suggesting the most likely correction. This makes it useful when you want a simple and direct word-level spell checker.

If `pyspellchecker` is not installed in your environment, install it first with `pip install pyspellchecker`.

In [18]:
from spellchecker import SpellChecker

spell = SpellChecker()

pyspell_examples = [
    "Introdaction",
    "electrcity",
    "langauage",
    "ths"
 ]

for word in pyspell_examples:
    print(f"{word} -> {spell.correction(word)}")

Introdaction -> introduction
electrcity -> electricity
langauage -> language
ths -> the


### Step 6: Tokenization with TextBlob

Tokenization means splitting text into smaller units (tokens), usually words.

`TextBlob` provides a simple way to tokenize a sentence into words using `.words`. This is useful before tasks such as frequency analysis, feature extraction, and classification.

In [25]:
tokenization_examples = pd.DataFrame(
    {
        "tweet": [
            "There would be less hype around AI and more action going forward.",
            "Python is the best tool!",
            "let's tokenize this sentence."
        ]
    }
)

# TextBlob tokenization: split each sentence into a list of word tokens
tokenization_examples["textblob_tokens"] = tokenization_examples["tweet"].apply(
    lambda text: list(TextBlob(text).words)
 )

tokenization_examples

,tweet,textblob_tokens
0,There would be less hype around AI and more ac...,"[There, would, be, less, hype, around, AI, and..."
1,Python is the best tool!,"[Python, is, the, best, tool]"
2,let's tokenize this sentence.,"[let, 's, tokenize, this, sentence]"


In [26]:
# Simple Python alternative (split) for comparison
mystring = "My favorite animal is wolf, it is cute, isn't it?"
mystring.split()

['My',
 'favorite',
 'animal',
 'is',
 'wolf,',
 'it',
 'is',
 'cute,',
 "isn't",
 'it?']

### Step 7: Stemming with NLTK

Stemming reduces words to their root form (or stem) by removing prefixes and suffixes.

Why stemming is useful:
- It normalizes similar word variations (for example: `running`, `runs` -> `run`).
- It helps text analysis focus on core meaning instead of surface form.
- It reduces vocabulary size, which can simplify NLP models and improve efficiency.

Limitations:
It can lead to overstemming or understemming, resulting in non-meaningful stems (e.g., "university" becomes "univers").

In [21]:
stemming_examples = pd.DataFrame(
    {
        "tweet": [
            "I like fishing",
            "I eat fish",
            "There are many fishes in pond"
        ]
    }
)

from nltk.stem import PorterStemmer
st = PorterStemmer()

stemming_examples["stemmed_tweet"] = stemming_examples["tweet"].apply(
    lambda text: " ".join(st.stem(word) for word in text.split())
 )

stemming_examples

,tweet,stemmed_tweet
0,I like fishing,i like fish
1,I eat fish,i eat fish
2,There are many fishes in pond,there are mani fish in pond


Famous stemmers:
- Porter Stemmer
- Snowball Stemmer (Porter2)
- ISRI Stemmer (for Arabic)

### Step 8: Lemmatization with TextBlob

A **lemma** is the dictionary base form of a word.

Examples:
- `running` -> `run`
- `better` -> `good` (in the right linguistic context)

Difference between stemming and lemmatization:
- **Stemming** cuts word endings using rules, and may produce non-dictionary forms.
- **Lemmatization** uses vocabulary and linguistic knowledge to return valid base words.

Because of this, lemmatization is usually more accurate, but can be a bit slower than stemming.

In [22]:
from textblob import Word

lemmatization_examples = pd.DataFrame(
    {
        "tweet": [
            "I like fishing",
            "I eat fish",
            "There are many fishes in pond",
            "leaves and leaf",
            "better and best"
        ]
    }
)

# Default lemmatization
lemmatization_examples["lemma_default"] = lemmatization_examples["tweet"].apply(
    lambda text: " ".join(Word(word).lemmatize() for word in text.split())
)

# Test-purpose demo only: here we force ALL words to adjective POS ("a").
# This helps illustrate cases like "better" -> "good".
lemmatization_examples["lemma_pos_aware"] = lemmatization_examples["tweet"].apply(
    lambda text: " ".join(Word(word).lemmatize("a") for word in text.split())
)

lemmatization_examples

,tweet,lemma_default,lemma_pos_aware
0,I like fishing,I like fishing,I like fishing
1,I eat fish,I eat fish,I eat fish
2,There are many fishes in pond,There are many fish in pond,There are many fishes in pond
3,leaves and leaf,leaf and leaf,leaves and leaf
4,better and best,better and best,good and best


NLTK supports lemmatization through `WordNetLemmatizer`.

Important note: without POS (part-of-speech), words like `better` may stay unchanged. To get `better -> good`, use adjective POS (`pos="a"`).

If the `wordnet` resource is missing, download it once in the current environment.

In [23]:
import nltk
from nltk.stem import WordNetLemmatizer

try:
    nltk.data.find("corpora/wordnet")
except LookupError:
    nltk.download("wordnet")

wnl = WordNetLemmatizer()

nltk_lemma_examples = ["running", "fishes", "better"]

# Without POS
default_lemmas = [wnl.lemmatize(word) for word in nltk_lemma_examples]

# With adjective POS to handle comparative adjectives like "better"
adj_lemmas = [wnl.lemmatize(word, pos="a") for word in nltk_lemma_examples]

default_lemmas, adj_lemmas

[nltk_data] Error loading wordnet: <urlopen error [WinError 10060] A
[nltk_data]     connection attempt failed because the connected party
[nltk_data]     did not properly respond after a period of time, or
[nltk_data]     established connection failed because connected host
[nltk_data]     has failed to respond>


(['running', 'fish', 'better'], ['running', 'fishes', 'good'])

## Summary of This Part

In this notebook so far, you learned how to:
- build a DataFrame for text data
- inspect core table properties
- drop duplicates
- apply practical Pandas text-cleaning functions
- preprocess text with lowercase conversion
- remove punctuation
- remove stop words
- standardize text
- correct spelling with TextBlob
- correct spelling with pyspellchecker
- tokenize text with TextBlob
- apply stemming with NLTK
- apply lemmatization with TextBlob (and NLTK)

This prepares the dataset for the next NLP steps such  feature extraction, and modeling.